# BSC - Mapa de Causalidade com Interdependencias Quantificaveis

**Empresa Ficticia de Tecnologia**

Este notebook transforma um BSC em um grafo de causalidade dirigido (DAG) com pesos estimados por regressao OLS.

---

## Estrutura

| Passo | O que fazemos | Conceito-chave |
|-------|---------------|----------------|
| **A** | Simular o dataset | Simulacao causal em ordem topologica |
| **B** | Grafo basico com NetworkX | DAG, nos e arestas |
| **C** | Estimar pesos por regressao | OLS padronizado por no |
| **D** | Grafo final com pesos | Visualizacao proporcional aos coeficientes |

## Variaveis do modelo

| Codigo | Variavel | Perspectiva |
|--------|----------|-------------|
| A1 | Horas de Treinamento | Aprendizado & Crescimento |
| A2 | Investimento em Tecnologia | Aprendizado & Crescimento |
| P1 | Eficiencia Operacional | Processos Internos |
| P2 | Taxa de Retrabalho | Processos Internos |
| C1 | Satisfacao do Cliente | Clientes |
| C2 | Taxa de Retencao | Clientes |
| F1 | Receita | Financeiro |
| F2 | Margem Operacional | Financeiro |

## Estrutura causal

```
A1 --+
     +--> P1 --> P2 --> C1 --> C2 --> F1 --> F2
A2 --+    +--------------------------->^
```

## Dependencias

Execute a linha abaixo se necessario:

In [ ]:
# !pip install numpy pandas matplotlib seaborn networkx scikit-learn

---
# PASSO A - Simulacao do Dataset

## Por que simular com estrutura causal?

Simular dados sem estrutura e trivial. O que queremos e que as **correlacoes emirjam naturalmente da estrutura causal**. Para isso, geramos as variaveis na **ordem topologica do grafo** -- nunca antes de gerar seus predecessores:

```
A1, A2 --> P1 --> P2, C1 --> C2 --> F1 --> F2
```

Cada variavel segue: `Y = intercepto + b1*X1 + b2*X2 + ruido`

## Por que padronizar (z-score)?

Variaveis em escalas diferentes (horas vs R$k vs indice 0-1) tornam os b incomparaveis. Padronizar (media=0, dp=1) resolve isso.

## Recovery test

Definimos b(A1->P1) = 0.38 na simulacao. No Passo C, o modelo deve estimar algo proximo. Essa e a validacao do metodo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

np.random.seed(42)
N = 200

# Paleta visual (dark theme)
DARK  = '#070910'
DARK2 = '#0d1017'
DARK3 = '#111620'
BORDER= '#1e2535'
TEXT  = '#e2e8f0'
MUTED = '#4a5568'
COLORS = {'A': '#3b82f6', 'P': '#8b5cf6', 'C': '#10b981', 'F': '#f59e0b'}

def node_color(col): return COLORS[col[0]]
def zscale(x): return (x - x.mean()) / x.std()

print('PASSO A - Simulacao do Dataset BSC')
print('=' * 40)

In [ ]:
# Perspectiva de Aprendizado & Crescimento
# Variaveis raiz: sem predecessores -> puro ruido gaussiano
A1 = np.random.normal(loc=32,  scale=6,   size=N)   # Horas de Treinamento
A2 = np.random.normal(loc=850, scale=150, size=N)   # Investimento em Tecnologia (R$k)

A1_s = zscale(A1)
A2_s = zscale(A2)

print(f'A1 (Horas Treinamento)   -> media: {A1.mean():.1f}h  | dp: {A1.std():.1f}h')
print(f'A2 (Invest. Tecnologia)  -> media: R${A2.mean():.0f}k | dp: R${A2.std():.0f}k')

In [ ]:
# Perspectiva de Processos Internos

# P1 (Eficiencia Operacional) <- A1, A2
# +0.38: mais treinamento -> mais eficiencia
# +0.31: mais tecnologia  -> mais eficiencia
P1 = (0.74
      + 0.38 * A1_s
      + 0.31 * A2_s
      + np.random.normal(0, 0.04, N))
P1 = np.clip(P1, 0.40, 0.99)

# P2 (Taxa de Retrabalho) <- P1
# beta NEGATIVO: mais eficiencia -> MENOS retrabalho
P2 = (0.35
      - 0.55 * zscale(P1)
      + np.random.normal(0, 0.02, N))
P2 = np.clip(P2, 0.01, 0.50)

print(f'P1 (Eficiencia Operacional) -> media: {P1.mean():.3f} | dp: {P1.std():.3f}')
print(f'P2 (Taxa de Retrabalho)     -> media: {P2.mean():.3f} | dp: {P2.std():.3f}')

In [ ]:
# Perspectiva de Clientes

# C1 (Satisfacao) <- P1 (+), P2 (-)
C1 = (0.30
      + 0.44 * zscale(P1)
      - 0.28 * zscale(P2)
      + np.random.normal(0, 0.03, N))
C1 = np.clip(C1, 0.30, 0.99)

# C2 (Retencao) <- C1 (+)
C2 = (0.20
      + 0.62 * zscale(C1)
      + np.random.normal(0, 0.03, N))
C2 = np.clip(C2, 0.30, 0.99)

print(f'C1 (Satisfacao do Cliente) -> media: {C1.mean():.3f} | dp: {C1.std():.3f}')
print(f'C2 (Taxa de Retencao)      -> media: {C2.mean():.3f} | dp: {C2.std():.3f}')

In [ ]:
# Perspectiva Financeira

F1 = (1500
      + 1800 * zscale(C2)
      + 900  * zscale(P1)
      + np.random.normal(0, 200, N))
F1 = np.clip(F1, 500, 8000)

F2 = (0.05
      + 0.71 * zscale(F1)
      + np.random.normal(0, 0.03, N))
F2 = np.clip(F2, 0.01, 0.60)

print(f'F1 (Receita)            -> media: R${F1.mean():.0f}k | dp: R${F1.std():.0f}k')
print(f'F2 (Margem Operacional) -> media: {F2.mean():.3f}     | dp: {F2.std():.3f}')

In [ ]:
# Montar DataFrame
df = pd.DataFrame({
    'A1': A1, 'A2': A2,
    'P1': P1, 'P2': P2,
    'C1': C1, 'C2': C2,
    'F1': F1, 'F2': F2,
})

print(f'Dataset: {df.shape[0]} observacoes x {df.shape[1]} variaveis')
df.describe().round(3)

### Verificacao: Matriz de Correlacao

Se a simulacao esta correta, devemos observar:
- `A1, A2` correlacao **positiva** com `P1`
- `P2` correlacao **negativa** com `P1` (beta = -0.55)
- `P2` correlacao **negativa** com `C1` (retrabalho prejudica satisfacao)
- `C1, C2, F1, F2` todos positivamente correlacionados entre si

In [ ]:
corr = df.corr().round(3)
print(corr)

In [ ]:
# Visualizacao diagnostica - 3 paineis
fig = plt.figure(figsize=(18, 14), facecolor=DARK)
gs  = gridspec.GridSpec(3, 2, figure=fig,
                        hspace=0.45, wspace=0.35,
                        left=0.07, right=0.96, top=0.93, bottom=0.06)

# Painel 1 - Heatmap correlacao (linha inteira)
ax_corr = fig.add_subplot(gs[0, :])
ax_corr.set_facecolor(DARK2)
cmap = sns.diverging_palette(220, 10, as_cmap=True)
sns.heatmap(corr, ax=ax_corr, annot=True, fmt='.2f',
            cmap=cmap, center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor=BORDER,
            annot_kws={'size': 9, 'color': TEXT},
            cbar_kws={'shrink': 0.8})
ax_corr.set_title('Matriz de Correlacao - verificacao da estrutura causal',
                  color=TEXT, fontsize=12, pad=12)
ax_corr.tick_params(colors=MUTED, labelsize=10)
ax_corr.collections[0].colorbar.ax.tick_params(colors=MUTED)
for lbl in ax_corr.get_xticklabels() + ax_corr.get_yticklabels():
    lbl.set_color(node_color(lbl.get_text()))
    lbl.set_fontweight('bold')

# Painel 2 - Histogramas (2x3)
inner_gs1 = gridspec.GridSpecFromSubplotSpec(2, 3, subplot_spec=gs[1, :],
                                              hspace=0.55, wspace=0.35)
for idx, col in enumerate(['A1', 'P1', 'P2', 'C1', 'F1', 'F2']):
    ax = fig.add_subplot(inner_gs1[idx // 3, idx % 3])
    ax.set_facecolor(DARK3)
    color = node_color(col)
    ax.hist(df[col], bins=30, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(col, color=color, fontsize=11, fontweight='bold', pad=6)
    ax.tick_params(colors=MUTED, labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(BORDER); spine.set_linewidth(0.5)
    ax.axvline(df[col].mean(), color='white', lw=1, ls='--', alpha=0.5,
               label=f'media={df[col].mean():.2f}')
    ax.legend(fontsize=7, facecolor=DARK2, edgecolor=BORDER, labelcolor=TEXT)

# Painel 3 - Scatterplots relacoes chave (1x3)
inner_gs2 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs[2, :],
                                              hspace=0.4, wspace=0.35)
scatter_pairs = [
    ('A1', 'P1', 'A1 -> P1  (beta=+0.38 esperado)'),
    ('P1', 'P2', 'P1 -> P2  (beta=-0.55 esperado)'),
    ('C1', 'C2', 'C1 -> C2  (beta=+0.62 esperado)'),
]
for idx, (xc, yc, title) in enumerate(scatter_pairs):
    ax = fig.add_subplot(inner_gs2[idx])
    ax.set_facecolor(DARK3)
    ax.scatter(df[xc], df[yc], alpha=0.35, s=18,
               color=node_color(xc), edgecolors='none')
    z = np.polyfit(df[xc], df[yc], 1)
    xl = np.linspace(df[xc].min(), df[xc].max(), 100)
    ax.plot(xl, np.poly1d(z)(xl), color='white', lw=1.5, alpha=0.8)
    r = df[[xc, yc]].corr().iloc[0, 1]
    ax.text(0.05, 0.93, f'r = {r:.3f}', transform=ax.transAxes,
            color=TEXT, fontsize=9, fontfamily='monospace',
            bbox=dict(facecolor=DARK2, edgecolor=BORDER, boxstyle='round,pad=0.3'))
    ax.set_xlabel(xc, color=node_color(xc), fontsize=9)
    ax.set_ylabel(yc, color=node_color(yc), fontsize=9)
    ax.set_title(title, color=TEXT, fontsize=9, pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(BORDER); spine.set_linewidth(0.5)

fig.suptitle('PASSO A - Diagnostico do Dataset Simulado',
             color=TEXT, fontsize=15, fontweight='bold', y=0.97)
plt.tight_layout()
plt.show()

In [ ]:
# Salvar dataset para os proximos passos
df.to_csv('bsc_dataset.csv', index=False)
print('Dataset salvo: bsc_dataset.csv')
print(f'Shape: {df.shape}')

---
# PASSO B - Grafo Basico com NetworkX

## O que e NetworkX?

NetworkX e a biblioteca padrao do Python para grafos. Armazena nos, arestas e atributos (pesos, cores, labels).

## DiGraph vs Graph

- `nx.Graph()` -- grafo **nao dirigido**: A-B e o mesmo que B-A
- `nx.DiGraph()` -- grafo **dirigido**: A->B e diferente de B->A

Usamos `DiGraph` porque causalidade tem direcao: treinamento causa eficiencia, nao o contrario.

## Layout manual

Definimos as coordenadas (x, y) de cada no explicitamente para criar uma progressao visual da esquerda (Aprendizado) para a direita (Financeiro).

In [ ]:
import networkx as nx
import matplotlib.patches as mpatches

# Definir o grafo
G = nx.DiGraph()

NODES = {
    'A1': 'Aprendizado', 'A2': 'Aprendizado',
    'P1': 'Processos',   'P2': 'Processos',
    'C1': 'Clientes',    'C2': 'Clientes',
    'F1': 'Financeiro',  'F2': 'Financeiro',
}
for node, persp in NODES.items():
    G.add_node(node, perspectiva=persp)

EDGES = [
    ('A1', 'P1'), ('A2', 'P1'),
    ('P1', 'P2'), ('P1', 'C1'),
    ('P2', 'C1'), ('C1', 'C2'),
    ('C2', 'F1'), ('P1', 'F1'),
    ('F1', 'F2'),
]
G.add_edges_from(EDGES)

print(f'Nos:     {list(G.nodes())}')
print(f'Arestas: {list(G.edges())}')
print(f'Total:   {G.number_of_nodes()} nos, {G.number_of_edges()} arestas')
print(f'E DAG?   {nx.is_directed_acyclic_graph(G)}')

In [ ]:
# Layout por perspectiva: x = profundidade causal, y = posicao vertical
pos = {
    'A1': (0, 2), 'A2': (0, 0),    # Aprendizado - coluna 0
    'P1': (2, 1), 'P2': (3, 0),    # Processos   - coluna 2-3
    'C1': (4, 2), 'C2': (5, 1),    # Clientes    - coluna 4-5
    'F1': (7, 2), 'F2': (7, 0),    # Financeiro  - coluna 7
}

COLOR_MAP = {
    'Aprendizado': '#3b82f6',
    'Processos':   '#8b5cf6',
    'Clientes':    '#10b981',
    'Financeiro':  '#f59e0b',
}
node_colors = [COLOR_MAP[NODES[n]] for n in G.nodes()]

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor(DARK)
ax.set_facecolor(DARK2)

zones = [
    ('Aprendizado\n& Crescimento', (-0.8, -0.7), 1.3, 3.4, '#3b82f611'),
    ('Processos\nInternos',        ( 1.2, -0.7), 2.5, 3.4, '#8b5cf611'),
    ('Clientes',                    ( 3.7, -0.7), 2.0, 3.4, '#10b98111'),
    ('Financeiro',                  ( 6.2, -0.7), 1.5, 3.4, '#f59e0b11'),
]
for label, (x, y), w, h, color in zones:
    ax.add_patch(plt.Rectangle((x, y), w, h, linewidth=0, facecolor=color, zorder=0))
    ax.text(x + w/2, y + h + 0.1, label, ha='center',
            color=MUTED, fontsize=8, style='italic')

nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=2000, alpha=0.95, ax=ax,
                       linewidths=1.5, edgecolors='#ffffff22')
nx.draw_networkx_labels(G, pos, font_color='white',
                        font_size=13, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#4a6fa5',
                       arrows=True, arrowsize=22, width=1.8, ax=ax,
                       connectionstyle='arc3,rad=0.05',
                       min_source_margin=28, min_target_margin=28)

legend_handles = [mpatches.Patch(color=c, label=p) for p, c in COLOR_MAP.items()]
ax.legend(handles=legend_handles, loc='lower right',
          facecolor=DARK3, edgecolor=BORDER, labelcolor=TEXT, fontsize=9)
ax.set_title('PASSO B - Grafo Causal Dirigido (sem pesos)', color=TEXT, fontsize=13, pad=16)
ax.set_xlim(-1, 8.5)
ax.set_ylim(-1.2, 3.8)
ax.axis('off')
plt.tight_layout()
plt.show()

---
# PASSO C - Estimando os Pesos por Regressao Linear (OLS)

## A logica

Para cada no com predecessores, fazemos uma regressao OLS:
```
Y = b0 + b1*X1 + b2*X2 + ... + e
```

Os coeficientes b estimados sao os **pesos das arestas**.

## Por que padronizar?

Padronizamos o dataset inteiro antes de regredir para que:
1. Os b sejam **comparaveis entre si** independente da escala original
2. Um b de 0.5 sempre signifique 'relacao moderada-forte'
3. Possamos ordenar arestas por forca causal sem vies de escala

## Recovery test

Definimos b(A1->P1) = 0.38 na simulacao. O modelo deve estimar algo proximo. Desvios pequenos (+-0.05) sao normais pelo ruido.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# Padronizar todo o dataset
scaler = StandardScaler()
df_s = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
)

print('Dataset padronizado - primeiras 3 linhas:')
print(df_s.head(3).round(3))

In [ ]:
# Estrutura causal: predecessores de cada no
PREDECESSORS = {
    'P1': ['A1', 'A2'],
    'P2': ['P1'],
    'C1': ['P1', 'P2'],
    'C2': ['C1'],
    'F1': ['C2', 'P1'],
    'F2': ['F1'],
}

# Betas de referencia (usados na simulacao - para comparar)
REF_BETAS = {
    'A1->P1': 0.38, 'A2->P1': 0.31, 'P1->P2': -0.55,
    'P1->C1': 0.44, 'P2->C1': -0.28, 'C1->C2': 0.62,
    'C2->F1': 0.49, 'P1->F1': 0.35,  'F1->F2': 0.71,
}

# Estimar regressao por no
edge_weights = {}
r2_scores    = {}

print('ESTIMACAO DOS PESOS - OLS PADRONIZADO')
print('=' * 50)

for target, sources in PREDECESSORS.items():
    X = df_s[sources].values
    y = df_s[target].values
    model = LinearRegression().fit(X, y)
    r2 = model.score(X, y)
    r2_scores[target] = r2

    print(f'\n  {target}  ~  {" + ".join(sources)}')
    print(f'  R2 = {r2:.3f}  ({r2*100:.1f}% da variancia explicada)')

    for source, coef in zip(sources, model.coef_):
        key = f'{source}->{target}'
        edge_weights[key] = round(coef, 3)
        ref = REF_BETAS.get(key, None)
        ref_str = f'  (ref: {ref:+.2f})' if ref else ''
        print(f'    beta({key}) = {coef:+.3f}{ref_str}')

In [ ]:
# Visualizacao: R2 e coeficientes
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor(DARK)

# Painel esquerdo: R2 por equacao
ax1 = axes[0]
ax1.set_facecolor(DARK2)
nodes_r2  = list(r2_scores.keys())
vals_r2   = list(r2_scores.values())
colors_r2 = [node_color(n) for n in nodes_r2]
bars = ax1.barh(nodes_r2, vals_r2, color=colors_r2, alpha=0.85, height=0.5)
ax1.axvline(0.7, color='white', lw=0.8, ls='--', alpha=0.4, label='limiar R2=0.70')
for bar, val in zip(bars, vals_r2):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', color=TEXT, fontsize=10, fontfamily='monospace')
ax1.set_xlim(0, 1.15)
ax1.set_xlabel('R2 (variancia explicada)', color=MUTED, fontsize=10)
ax1.set_title('R2 por equacao de regressao', color=TEXT, fontsize=12, pad=10)
ax1.tick_params(colors=TEXT, labelsize=11)
ax1.legend(facecolor=DARK3, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
for spine in ax1.spines.values(): spine.set_color(BORDER)

# Painel direito: coeficientes beta
ax2 = axes[1]
ax2.set_facecolor(DARK2)
edges_list = list(edge_weights.keys())
betas      = list(edge_weights.values())
colors_b   = ['#ef4444' if b < 0 else '#4a9eff' for b in betas]
bars2 = ax2.barh(edges_list, betas, color=colors_b, alpha=0.85, height=0.5)
ax2.axvline(0, color=TEXT, lw=0.8, alpha=0.4)
for bar, val in zip(bars2, betas):
    xpos = val + 0.015 if val >= 0 else val - 0.015
    ha   = 'left' if val >= 0 else 'right'
    ax2.text(xpos, bar.get_y() + bar.get_height()/2,
             f'{val:+.3f}', va='center', ha=ha,
             color=TEXT, fontsize=9, fontfamily='monospace')
ax2.set_xlabel('Coeficiente beta (padronizado)', color=MUTED, fontsize=10)
ax2.set_title('Pesos estimados por aresta - beta OLS padronizado',
              color=TEXT, fontsize=12, pad=10)
ax2.tick_params(colors=TEXT, labelsize=9)
ax2.legend(handles=[
    mpatches.Patch(color='#4a9eff', label='beta positivo'),
    mpatches.Patch(color='#ef4444', label='beta negativo'),
], facecolor=DARK3, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
for spine in ax2.spines.values(): spine.set_color(BORDER)

fig.suptitle('PASSO C - Estimacao dos Pesos Causais por OLS',
             color=TEXT, fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
# PASSO D - Grafo Final com Pesos nas Arestas

## O que muda em relacao ao Passo B?

Agora cada aresta tem um **peso beta estimado** pelo modelo:

- **Espessura** da aresta proporcional a |beta| -- relacoes mais fortes aparecem mais grossas
- **Cor azul** = beta positivo (causa aumenta o efeito)
- **Cor vermelha** = beta negativo (causa diminui o efeito -- ex: P1->P2)
- **Label** em cada aresta mostra o valor exato de beta

## Leitura dos coeficientes

- `beta(C1->C2) ~ +0.62` -> satisfacao e o principal motor de retencao
- `beta(F1->F2) ~ +0.71` -> receita e o principal determinante da margem
- `beta(P1->P2) ~ -0.55` -> eficiencia reduz fortemente o retrabalho
- `beta(A1->P1) ~ +0.38` -> treinamento tem impacto significativo na eficiencia

In [ ]:
# Adicionar pesos ao grafo
for edge_key, weight in edge_weights.items():
    src, tgt = edge_key.split('->')
    G[src][tgt]['weight'] = weight

# Atributos visuais das arestas
edge_list   = list(G.edges())
edge_betas  = [G[u][v]['weight'] for u, v in edge_list]
edge_widths = [abs(b) * 7 + 0.8 for b in edge_betas]
edge_colors = ['#ef4444' if b < 0 else '#4a9eff' for b in edge_betas]

edge_labels = {
    (u, v): f"{G[u][v]['weight']:+.2f}"
    for u, v in G.edges()
}

# Figura
fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor(DARK)
ax.set_facecolor(DARK2)

# Zonas de fundo
zones = [
    ('Aprendizado\n& Crescimento', (-0.9, -0.8), 1.4, 3.6, '#3b82f60d'),
    ('Processos\nInternos',        ( 1.1, -0.8), 2.6, 3.6, '#8b5cf60d'),
    ('Clientes',                    ( 3.6, -0.8), 2.1, 3.6, '#10b9810d'),
    ('Financeiro',                  ( 6.1, -0.8), 1.6, 3.6, '#f59e0b0d'),
]
for label, (x, y), w, h, color in zones:
    ax.add_patch(plt.Rectangle((x, y), w, h, linewidth=0.5,
                               edgecolor=BORDER, facecolor=color, zorder=0))
    ax.text(x + w/2, y + h + 0.08, label, ha='center',
            color=MUTED, fontsize=8, style='italic', linespacing=1.4)

node_colors = [COLOR_MAP[NODES[n]] for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=2400, alpha=0.95, ax=ax,
                       linewidths=2, edgecolors='#ffffff18')
nx.draw_networkx_labels(G, pos, font_color='white',
                        font_size=13, font_weight='bold', ax=ax)

for (u, v), width, color in zip(edge_list, edge_widths, edge_colors):
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)],
                           edge_color=color, width=width, alpha=0.75,
                           arrows=True, arrowsize=20, ax=ax,
                           connectionstyle='arc3,rad=0.06',
                           min_source_margin=30, min_target_margin=30)

nx.draw_networkx_edge_labels(
    G, pos, edge_labels=edge_labels,
    font_size=8, font_color='#cbd5e1',
    bbox=dict(boxstyle='round,pad=0.25', facecolor=DARK3,
              edgecolor=BORDER, alpha=0.9),
    ax=ax
)

legend_handles = [mpatches.Patch(color=c, label=p) for p, c in COLOR_MAP.items()]
legend_handles += [
    mpatches.Patch(color='#4a9eff', label='beta positivo (causa aumenta efeito)'),
    mpatches.Patch(color='#ef4444', label='beta negativo (causa diminui efeito)'),
]
ax.legend(handles=legend_handles, loc='lower right', ncol=2,
          facecolor=DARK3, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)
ax.set_title('PASSO D - BSC: Grafo Causal com Pesos Estimados (OLS Padronizado)',
             color=TEXT, fontsize=13, pad=18)
ax.set_xlim(-1.0, 8.8)
ax.set_ylim(-1.2, 3.9)
ax.axis('off')
plt.tight_layout()
plt.show()

---
# Resumo Final

| Passo | Entregavel | Conceito central |
|-------|-----------|------------------|
| **A** | Dataset simulado 200 obs x 8 vars | Simulacao causal em ordem topologica |
| **B** | Grafo DAG basico com NetworkX | Nos, arestas, layout por perspectiva |
| **C** | Coeficientes beta estimados por OLS | Regressao padronizada por no |
| **D** | Grafo completo com pesos visuais | Espessura proporcional a |beta|, cor por sinal |

## Proximas evolucoes

1. **SEM (Structural Equation Modeling)** com `semopy` -- estima todos os caminhos simultaneamente com indices de ajuste (CFI, RMSEA)
2. **Efeitos indiretos** -- impacto total de A1 em F1 via multiplos caminhos
3. **Simulacao de cenarios** -- se A1 aumentar 10%, qual o impacto em F2?
4. **Dados reais** -- substituir a simulacao por dados historicos da organizacao